# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Sztenberg, Ian Ezequiel**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [6]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

#Agregado
from aima_libs.aima import PriorityQueue as AimaPriorityQueue

In [ ]:
def priority_function(node):
    """
    Función que define la función f()=costo+heurísitica , que utilizamos para asignar prioridad al node al guardarlo en la cola
    """
    righ_rod = 2
    sol = [5,4,3,2,1]
    h = 0
    #costo del nodo

    g = node.state.accumulated_cost
    #herística del nodo
    #
    #print(node.state.get_state()[righ_rod])
    for i in  range(len(node.state.get_state()[righ_rod])):
        if node.state.get_state()[righ_rod][i] == sol[i]:
            h-=1
            

    #print(g)

    return (g+h)


In [125]:
#node = NodeHanoi(StatesHanoi([3,2,1],[],[5,4],5))
node = NodeHanoi(StatesHanoi([],[2],[5,4,3,1],5))
f = priority_function(node)
print(f)



-3.0


In [128]:
def a_star_search(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    #A* necesita cola prioritaria. El más prioritario, es el número más bajo
    queue = AimaPriorityQueue(order='min',f=priority_function)

    #agregamos estado inicial a la cola
    queue.append(NodeHanoi(problem.initial))

    #contador de nodos explorados
    nodes_explored = 0

    #set de estados ya explorados
    explored = set()

    #depth alcanzado
    depth_achieved = 0

    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    metrics = {
        "solution_found": False,
        "nodes_explored": None,
        "states_visited": None,
        "nodes_in_frontier": None,
        "max_depth": None,
        "cost_total": None,
    }
    solution = NodeHanoi(initial_state)

    #mientras la lista no esté vacía
    while len(queue)!=0:
        #tomamos el nodo de menor prioridad
        priority,node = queue.pop()
        #incrementamos contador de nodos explorados
        nodes_explored+=1

        print(f"Sequence {nodes_explored}. node: {node}. Priority {priority}")

        #evaluo depth alcanzado
        if node.depth>depth_achieved:
            depth_achieved=node.depth
        
        #agregamos estado a set de ya explorados
        explored.add(node.state)
        
        if problem.goal_test(node.state):
            #si es solución
            metrics = {
                "solution_fount": True,
                "nodes_explored": nodes_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(queue),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node,metrics
        
        #si no es solución, expandimos el estado
        for next_node in node.expand(problem):
            if next_node.state not in explored:
                #guardamos nuevo estado en la cola 
                queue.append(next_node)
        

    metrics = {
        "solution_fount": False,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(queue),
        "max_depth": depth_achieved,
        "cost_total": None,
    }


    return solution, metrics

Se prueba la función:

In [135]:
#%%timeit
solution, metrics = a_star_search(number_disks=5)

Sequence 1. node: <Node HanoiState: 5 4 3 2 1 |  | >. Priority 0.0
Sequence 2. node: <Node HanoiState: 5 4 3 2 | 1 | >. Priority 1.0
Sequence 3. node: <Node HanoiState: 5 4 3 2 |  | 1>. Priority 1.0
Sequence 4. node: <Node HanoiState: 5 4 3 | 1 | 2>. Priority 2.0
Sequence 5. node: <Node HanoiState: 5 4 3 2 |  | 1>. Priority 2.0
Sequence 6. node: <Node HanoiState: 5 4 3 | 2 | 1>. Priority 2.0
Sequence 7. node: <Node HanoiState: 5 4 3 1 |  | 2>. Priority 3.0
Sequence 8. node: <Node HanoiState: 5 4 3 | 2 | 1>. Priority 3.0
Sequence 9. node: <Node HanoiState: 5 4 3 | 2 1 | >. Priority 3.0
Sequence 10. node: <Node HanoiState: 5 4 3 |  | 2 1>. Priority 3.0
Sequence 11. node: <Node HanoiState: 5 4 3 1 | 2 | >. Priority 3.0
Sequence 12. node: <Node HanoiState: 5 4 3 1 | 2 | >. Priority 4.0
Sequence 13. node: <Node HanoiState: 5 4 | 2 1 | 3>. Priority 4.0
Sequence 14. node: <Node HanoiState: 5 4 3 1 | 2 | >. Priority 4.0
Sequence 15. node: <Node HanoiState: 5 4 | 3 | 2 1>. Priority 4.0
Sequence

Veamos las métricas:

In [136]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_fount: True
nodes_explored: 268
states_visited: 169
nodes_in_frontier: 18
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [109]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [110]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3


In [137]:
solution.generate_solution_for_simulator()